In [6]:
import os
from pathlib import Path
from typing import List

from tqdm import tqdm
import numpy as np
import pandas as pd
try:
    import google.generativeai as genai
except ImportError as exc:
    raise ImportError("Install google-generativeai via `pip install google-generativeai`.") from exc


In [7]:
# ---- Configuration ----
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
if GOOGLE_API_KEY is not None:
    GEMINI_API_KEY = GOOGLE_API_KEY
if not GEMINI_API_KEY:
    raise EnvironmentError("Set GEMINI_API_KEY in your environment before running this cell.")

genai.configure(api_key=GEMINI_API_KEY)

ROOT_DIR = Path("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/repo_callgraph_clusters").expanduser()
if not ROOT_DIR.exists():
    raise FileNotFoundError(f"Root directory not found: {ROOT_DIR}")

MODEL_NAME = "text-embedding-004"
BATCH_SIZE = 150  # number of .py files to embed before logging progress


In [8]:
def read_python_file(file_path: Path) -> str:
    """Return the contents of a Python file with a lightweight header."""
    return f"# File: {file_path.name}\n" + file_path.read_text(encoding="utf-8", errors="ignore")


def embed_text(texts: List[str]) -> List[float]:
    response = genai.embed_content(
        model=MODEL_NAME,
        content=texts,
        task_type="SEMANTIC_SIMILARITY",
    )
    return response["embedding"]


def embed_python_file(file_path: Path) -> np.ndarray:
    source = read_python_file(file_path)
    if not source.strip():
        raise ValueError(f"{file_path} is empty or unreadable")
    return np.asarray(embed_text(source), dtype=np.float32)


In [12]:
records = []
pattern_dirs = [p for p in sorted(ROOT_DIR.iterdir()) if p.is_dir()]
if not pattern_dirs:
    raise ValueError(f"No pattern directories detected in {ROOT_DIR}")

python_files = []
for pattern_dir in pattern_dirs:
    python_files.extend((pattern_dir.name, py_file) for py_file in sorted(pattern_dir.glob("**/*.py")))

if not python_files:
    raise ValueError("No .py files found under the supplied root directory.")

total_files = len(python_files)
for batch_start in range(0, total_files, BATCH_SIZE):
    batch = python_files[batch_start : batch_start + BATCH_SIZE]
    print(
        f"Processing files {batch_start + 1}-{batch_start + len(batch)} / {total_files}"
    )

    texts = [read_python_file(py_path) for _, py_path in batch]
    embeddings = embed_text(texts)
    for (pattern_name, py_path), embedding_vector in zip(batch, embeddings):
        records.append(
            {
                "pattern": pattern_name,
                "file": str(py_path.relative_to(ROOT_DIR)),
                "embedding": embedding_vector,
            }
        )
    
    # for pattern_name, py_path in tqdm(batch, desc="Files", leave=False):
    #     try:
    #         embedding_vector = embed_python_file(py_path)
    #     except ValueError as err:
    #         print(f"Skipping {py_path}: {err}")
    #         continue
    #     records.append(
    #         {
    #             "pattern": pattern_name,
    #             "file": str(py_path.relative_to(ROOT_DIR)),
    #             "embedding": embedding_vector,
    #         }
    #     )

if not records:
    raise ValueError("No embeddings were generated; ensure .py files have content.")

embedding_length = len(records[0]["embedding"])
rows = []
for record in records:
    row = {f"dim_{i+1}": value for i, value in enumerate(record["embedding"])}
    row["pattern"] = record["pattern"]
    row["file"] = record["file"]
    rows.append(row)

embeddings_df = pd.DataFrame(rows)
print(
    f"Generated embeddings for {len(embeddings_df)} files across {len(pattern_dirs)} patterns with {embedding_length} dimensions."
)
display(embeddings_df.head())

OUTPUT_PATH = Path("./results/pattern_embeddings/gemini_communities_embedding.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
embeddings_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved embeddings to {OUTPUT_PATH.resolve()}")


Processing files 1-150 / 2442
Processing files 151-300 / 2442
Processing files 151-300 / 2442
Processing files 301-450 / 2442
Processing files 301-450 / 2442
Processing files 451-600 / 2442
Processing files 451-600 / 2442
Processing files 601-750 / 2442
Processing files 601-750 / 2442
Processing files 751-900 / 2442
Processing files 751-900 / 2442
Processing files 901-1050 / 2442
Processing files 901-1050 / 2442
Processing files 1051-1200 / 2442
Processing files 1051-1200 / 2442
Processing files 1201-1350 / 2442
Processing files 1201-1350 / 2442
Processing files 1351-1500 / 2442
Processing files 1351-1500 / 2442
Processing files 1501-1650 / 2442
Processing files 1501-1650 / 2442
Processing files 1651-1800 / 2442
Processing files 1651-1800 / 2442
Processing files 1801-1950 / 2442
Processing files 1801-1950 / 2442
Processing files 1951-2100 / 2442
Processing files 1951-2100 / 2442
Processing files 2101-2250 / 2442
Processing files 2101-2250 / 2442
Processing files 2251-2400 / 2442
Proces

,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10,...,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767,dim_768,pattern,file
0,0.016026,-0.013758,-0.037220,0.032744,0.052628,0.009569,0.058647,-0.023813,0.011639,-0.004973,...,0.021485,0.051842,0.012327,0.023157,-0.037828,-0.008317,0.092225,-0.028479,3DOD_thesis,3DOD_thesis/cluster_0.py
1,-0.007504,-0.003126,-0.020998,0.022419,0.042379,0.025537,0.040804,0.006903,0.017828,-0.018028,...,0.008884,0.045681,0.006773,0.026314,-0.052060,-0.040376,0.069306,-0.020113,3DOD_thesis,3DOD_thesis/cluster_1.py
2,-0.000291,-0.004054,-0.040742,0.022507,0.048653,0.006458,0.038025,-0.016801,0.024710,0.007949,...,0.013546,0.046916,0.018118,0.019254,-0.050418,-0.024308,0.083873,-0.011763,3DOD_thesis,3DOD_thesis/cluster_10.py
3,0.011949,-0.014774,-0.029098,0.052298,0.040381,0.050559,0.038567,0.023808,-0.008616,-0.022944,...,-0.000861,0.034087,0.008946,0.002147,-0.035787,-0.026105,0.082386,-0.051664,3DOD_thesis,3DOD_thesis/cluster_2.py
4,0.053531,-0.010405,-0.031947,0.037319,0.074302,0.013438,0.062306,-0.012757,-0.015362,-0.020760,...,0.031407,0.007661,-0.040618,0.019052,-0.020214,0.004445,0.060633,-0.075090,3DOD_thesis,3DOD_thesis/cluster_3.py


Saved embeddings to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_communities_embedding.csv


In [ ]:
embeddings_df.shape